##  Ingest results.json files
### Read the file using spark dataframe reader API
### Add Meta Data Columns
- Source File
- Ingestion Timestamp
### Write to Bronze delta table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-help

In [0]:
source_file=f"{landing_folder_path}/{v_batch_id}/results"
table_name=f"{catalog_name}.{bronze_schema}.results"

###  Step 1- Read the csv file using dataframe reader API

In [0]:
#define the schema
from pyspark.sql.types import StructType,StructField,StringType,DateType,IntegerType,FloatType
results_schema=StructType([
    StructField('date',DateType()),
    StructField('raceName',StringType()),
    StructField('round',IntegerType()),
    StructField('season',IntegerType()),
     StructField('url',StringType()),
    StructField('constructorId',StringType()),
    StructField('driverId',StringType()),
   StructField('grid',IntegerType()),
    StructField('laps',IntegerType()),
    StructField('number',IntegerType()),
    StructField('points',FloatType()),
    StructField('position',IntegerType()),
    StructField('positionText',StringType()),
    StructField('status',StringType())
])

In [0]:
#spark reader API
results_df=(spark.read
                .format('json')
                .schema(results_schema)
                .option("mode","failFast")
                .load(source_file)
                )


### Step 2 Add Meta Data Columns
- Source File
- Ingestion Timestamp

In [0]:
results_final_df=add_ingestion_metadata(results_df)

### Step 3 Write to Bronze delta table

In [0]:
write_to_bronze (
    input_df = results_final_df,
    target_table= table_name,
    batch_id=v_batch_id
)

In [0]:
display(spark.table(table_name))